# TinyML Food Classification Pipeline (Food-101 ➜ Arduino UNO R4)

This notebook is **intentionally verbose and transparent**. Every major block answers:
- **What is happening?**
- **Why are we doing it?**
- **How can you verify correctness?**

The goal is not just accuracy; it is understanding and verification across the full TinyML workflow: data handling, preprocessing, modeling, evaluation, overfitting diagnosis, and embedded deployment constraints.


## 0) Environment & Reproducibility

We set seeds for reproducibility. This does **not** guarantee identical results across hardware/versions, but it reduces randomness.


In [ ]:
import os
import random
import json
import math
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)


## 1) Acquire the Food-101 dataset (public TFDS source)

**What is happening?**
We use TensorFlow Datasets (TFDS) to download the Food-101 dataset. This dataset is more varied and less "clean" than Fruits-360, which helps avoid trivial overfitting.

**Why?**
We want a dataset that is *harder to overfit* and closer to real-world conditions (diverse lighting, backgrounds, and camera angles).

**How to verify?**
We print dataset metadata, class names, and confirm the train/test splits.


In [ ]:
(train_raw, test_raw), info = tfds.load(
    "food101",
    split=["train", "validation"],
    as_supervised=True,
    with_info=True,
)

print(info)
print("Number of classes:", info.features["label"].num_classes)


## 2) Choose an 8-class subset transparently

**What is happening?**
We select an **explicit** list of eight classes from Food-101. If any are missing, we stop and show the mismatch.

**Why?**
A small subset makes the TinyML model and training process tractable and more interpretable.

**How to verify?**
We list all available classes and show which are selected.


In [ ]:
ALL_CLASSES = info.features["label"].names
print(f"Total classes found: {len(ALL_CLASSES)}")

SELECTED_CLASSES = [
    "apple_pie",
    "carrot_cake",
    "chocolate_cake",
    "french_toast",
    "ice_cream",
    "pancakes",
    "strawberry_shortcake",
    "waffles",
]

missing = [c for c in SELECTED_CLASSES if c not in ALL_CLASSES]
if missing:
    print("Missing classes detected:", missing)
    print("First 20 available classes:", ALL_CLASSES[:20])
    # Deterministic fallback to avoid hard failure in student environments
    SELECTED_CLASSES = ALL_CLASSES[:8]
    print("Falling back to first 8 classes:", SELECTED_CLASSES)

print("Selected classes:", SELECTED_CLASSES)


In [ ]:
# Map selected class names to indices
selected_indices = np.array([ALL_CLASSES.index(c) for c in SELECTED_CLASSES], dtype=np.int64)
print("Selected class indices:", selected_indices)

# Lookup table for remapping labels to 0..7
lookup = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(selected_indices, dtype=tf.int64),
        values=tf.range(len(selected_indices), dtype=tf.int64),
    ),
    default_value=-1,
)


## 3) Filter and remap to the selected classes

**What is happening?**
We filter the full dataset down to the eight selected classes, then remap labels from the original 0–100 range to 0–7.

**Why?**
This keeps the model small and avoids unnecessary outputs.

**How to verify?**
We count examples per class in train and test after filtering.


In [ ]:
def filter_selected(image, label):
    return tf.reduce_any(tf.equal(label, selected_indices))

def remap_label(image, label):
    new_label = lookup.lookup(label)
    return image, new_label

train_filtered = train_raw.filter(filter_selected).map(remap_label)
test_filtered = test_raw.filter(filter_selected).map(remap_label)

# Count examples per class

def count_by_class(ds, num_classes):
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, label in tfds.as_numpy(ds):
        counts[int(label)] += 1
    return counts

train_counts = count_by_class(train_filtered, len(SELECTED_CLASSES))
test_counts = count_by_class(test_filtered, len(SELECTED_CLASSES))

counts_df = pd.DataFrame({
    "class": SELECTED_CLASSES,
    "train": train_counts,
    "test": test_counts,
})
counts_df


**Verification checklist:**
- Each selected class should have non-zero counts.
- Train/test counts should be reasonably balanced.


## 4) Raw vs. preprocessed data (transparent inspection)

**What is happening?**
We show **raw images** directly from TFDS, then show **preprocessed** images after resizing and normalization.

**Why?**
Students must see the transformation steps and understand that preprocessing changes the input distribution.

**How to verify?**
We sample images randomly (not by batch) to avoid repeated/similar visuals.


In [ ]:
IMG_SIZE = (64, 64)

rng = np.random.default_rng(SEED)

# Random unbatched sampling for raw images
raw_samples = []
for cls_idx, cls_name in enumerate(SELECTED_CLASSES[:4]):
    # grab one example of this class
    sample = next(iter(train_filtered.filter(lambda x, y: y == cls_idx).take(1)))
    raw_samples.append((cls_name, sample[0]))

plt.figure(figsize=(8, 6))
for i, (cls, img) in enumerate(raw_samples, 1):
    plt.subplot(2, 2, i)
    plt.imshow(img.numpy())
    plt.title(f"Raw: {cls}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Preprocessed view (resize + normalization visualization)
plt.figure(figsize=(8, 6))
for i, (cls, img) in enumerate(raw_samples, 1):
    img_resized = tf.image.resize(img, IMG_SIZE)
    arr = img_resized.numpy() / 255.0
    plt.subplot(2, 2, i)
    plt.imshow(arr)
    plt.title(f"Preprocessed: {cls}")
    plt.axis("off")
plt.tight_layout()
plt.show()


**Why not rely on batch visualizations?**
Batch sampling can show repeated or very similar images because batch pipelines may shuffle within a limited buffer. Using unbatched random sampling gives a clearer view of diversity.


## 5) Build TensorFlow datasets (train/val/test)

**What is happening?**
We split the filtered training data into **train** and **validation**, while keeping the official test split untouched.

**Why?**
Validation guides training decisions; the test set is reserved for the final honest evaluation.

**How to verify?**
We print sample counts and shapes.


In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.2

# Count total train samples after filtering
train_total = int(train_counts.sum())
val_size = int(train_total * VAL_SPLIT)

train_filtered_shuffled = train_filtered.shuffle(1000, seed=SEED, reshuffle_each_iteration=False)
val_ds = train_filtered_shuffled.take(val_size)
train_ds = train_filtered_shuffled.skip(val_size)

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, tf.one_hot(label, depth=len(SELECTED_CLASSES))

train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_filtered.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train samples: {train_total - val_size}")
print(f"Val samples: {val_size}")
print(f"Test samples: {int(test_counts.sum())}")


### 5.1) Data augmentation (training only)

**What is happening?**
We apply augmentation **only** to the training set. Validation and test remain untouched.

**Why?**
Augmentation helps generalization but must not contaminate evaluation.

**How to verify?**
We visualize augmented images and note their distortions.


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

autotune = tf.data.AUTOTUNE
augmented_train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=autotune)


In [ ]:
# Visualize augmented images (note the distortions)
plt.figure(figsize=(8, 6))
for images, labels in augmented_train_ds.take(1):
    for i in range(4):
        plt.subplot(2, 2, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title("Augmented")
        plt.axis("off")
plt.tight_layout()
plt.show()


**Note:** Augmentations can make images look "unnatural". This is expected and is part of learning robustness.


## 6) Tiny CNN architecture (Arduino UNO R4 realistic)

**What is happening?**
We build a **tiny** CNN using separable convolutions and global average pooling.

**Why?**
These layers drastically reduce parameter count and computation while preserving accuracy.

**How to verify?**
We print the model summary and parameter count.


In [ ]:
num_classes = len(SELECTED_CLASSES)

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(*IMG_SIZE, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.SeparableConv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.SeparableConv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.SeparableConv2D(64, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

param_count = model.count_params()
print(f"Total parameters: {param_count}")


## 7) Train with safeguards against overfitting

**What is happening?**
We train with early stopping and learning-rate scheduling.

**Why?**
To avoid wasting epochs and to reduce overfitting.

**How to verify?**
We plot accuracy and loss for both train and validation.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss')
]

history = model.fit(
    augmented_train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Plot training curves
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['accuracy'], label='Train Acc')
plt.plot(history_df['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df['loss'], label='Train Loss')
plt.plot(history_df['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()


**Interpretation guide:**
- If training accuracy climbs while validation accuracy stalls or drops, overfitting is emerging.
- If both are low, underfitting persists.
- If validation is extremely high (e.g., ~100%), be skeptical; check for leakage and dataset bias.


## 8) Consolidated evaluation (single block)

**What is happening?**
We compute accuracy, precision, recall, F1 (macro & weighted), a per-class report, and a confusion matrix **only on the test set**.

**Why?**
Accuracy alone can hide class imbalance or systematic mistakes, and test data should remain untouched until the end.

**How to verify?**
We print metrics and visualize a full confusion matrix with all classes.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score

# Get predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy: {acc:.4f}")
print(f"Macro Precision/Recall/F1: {precision_macro:.4f} / {recall_macro:.4f} / {f1_macro:.4f}")
print(f"Weighted Precision/Recall/F1: {precision_weighted:.4f} / {recall_weighted:.4f} / {f1_weighted:.4f}")

print("
Classification Report:")
print(classification_report(y_true, y_pred, target_names=SELECTED_CLASSES, zero_division=0))

# Confusion matrix (always show full matrix)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(SELECTED_CLASSES))))

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(np.arange(len(SELECTED_CLASSES)), SELECTED_CLASSES, rotation=45, ha='right')
plt.yticks(np.arange(len(SELECTED_CLASSES)), SELECTED_CLASSES)

thresh = cm.max() / 2 if cm.max() > 0 else 1
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()


## 9) Overfitting & validity checks

Perfect or near-perfect results on real-world datasets are **suspicious**. Food-101 is more diverse than Fruits-360, but it can still exhibit dataset bias.

We run two checks:
1) **Stress-test augmentations** to probe prediction stability.
2) **Duplicate-image detection** on a subset to check for accidental overlap across splits.


In [ ]:
# 9.1) Stress-test augmentation

stress_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.25),
    tf.keras.layers.RandomContrast(0.3),
])

# Take a small random subset from test set
sample_images = []
sample_labels = []
for images, labels in test_ds.take(3):
    sample_images.append(images)
    sample_labels.append(labels)

sample_images = tf.concat(sample_images, axis=0)
aug_images = stress_aug(sample_images, training=True)

preds_original = model.predict(sample_images, verbose=0)
preds_aug = model.predict(aug_images, verbose=0)

instability = np.mean(np.argmax(preds_original, axis=1) != np.argmax(preds_aug, axis=1))
print(f"Prediction instability under strong augmentation: {instability:.2%}")


In [ ]:
# 9.2) Duplicate-image hash check on a subset

# Hashing every image is expensive; we hash a small sample to detect obvious duplication.

def image_hash(img):
    img_bytes = tf.io.encode_jpeg(tf.cast(img, tf.uint8)).numpy()
    return hashlib.md5(img_bytes).hexdigest()

train_sample_hashes = set()
for images, _ in train_ds.take(5):
    for img in images:
        train_sample_hashes.add(image_hash(img))

duplicate_count = 0
for images, _ in test_ds.take(5):
    for img in images:
        if image_hash(img) in train_sample_hashes:
            duplicate_count += 1

print(f"Duplicate hashes found in sample: {duplicate_count}")


**Interpretation:**
- High instability under strong augmentation suggests the model is brittle.
- Non-zero duplicate hashes in the sample may indicate leakage or near-duplicates.


## 10) INT8 TensorFlow Lite conversion (Arduino-ready)

**What is happening?**
We convert the model to **full INT8** and verify predictions against the original Keras model.

**Why?**
Arduino UNO R4 has limited RAM/flash. INT8 reduces size and speeds inference.

**How to verify?**
We compare predictions on a small test sample between Keras and TFLite.


In [ ]:
# Representative dataset for quantization

def representative_dataset():
    for images, _ in train_ds.take(10):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
print("TFLite conversion successful. Size (bytes):", len(tflite_model))


In [ ]:
# Validate TFLite predictions vs Keras on a small batch

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# Grab a small batch
images, labels = next(iter(test_ds.take(1)))

# Keras predictions
keras_preds = model.predict(images, verbose=0)
keras_top = np.argmax(keras_preds, axis=1)

# TFLite predictions
input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]

# Quantize input
images_int8 = (images / input_scale + input_zero_point).numpy().astype(np.int8)

interpreter.set_tensor(input_details['index'], images_int8)
interpreter.invoke()

output_int8 = interpreter.get_tensor(output_details['index'])
# Dequantize output
output_float = (output_int8.astype(np.float32) - output_zero_point) * output_scale

tflite_top = np.argmax(output_float, axis=1)

agreement = np.mean(keras_top == tflite_top)
print(f"Keras vs TFLite top-1 agreement: {agreement:.2%}")


### 10.1) Export TFLite model for Arduino conversion

The resulting `tflite_model` can be saved and later converted into a C header for Arduino deployment.


In [ ]:
TFLITE_PATH = Path("model_int8.tflite")
TFLITE_PATH.write_bytes(tflite_model)
print("Saved:", TFLITE_PATH)


## 11) Final reflection: Why skepticism matters

Even with strong metrics, Food-101 is still a controlled dataset. Real-world foods vary in lighting, occlusion, plating, and backgrounds.

This notebook demonstrates:
- Transparent data handling
- Deliberate class selection
- Proper augmentation usage
- Model design tailored to TinyML constraints
- Honest evaluation and overfitting checks
- Full INT8 conversion suitable for Arduino UNO R4

Use this as a rigorous starting point, not an endpoint.
